In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. Load the dataset
df = pd.read_csv('autism_screening.csv')

# 2. Data Cleaning
# Handle '?' values and outliers
df['ethnicity'] = df['ethnicity'].replace('?', 'Others')
df.loc[df['age'] > 100, 'age'] = df['age'].median()
df['age'] = df['age'].fillna(df['age'].median())

# 3. Feature Selection
# We use A1-A10 scores, age, gender, jaundice history, and family autism history
features = ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score',
            'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score',
            'age', 'gender', 'jundice', 'austim']

X = df[features].copy()
y = df['Class/ASD'].apply(lambda x: 1 if x == 'YES' else 0)

# 4. Encoding Categorical Variables
# gender: f=0, m=1 | jaundice: no=0, yes=1 | austim: no=0, yes=1
X['gender'] = LabelEncoder().fit_transform(X['gender'])
X['jundice'] = LabelEncoder().fit_transform(X['jundice'])
X['austim'] = LabelEncoder().fit_transform(X['austim'])

# 5. Train the Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Save the Model
with open('autism_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model trained and saved as 'autism_model.pkl'")

FileNotFoundError: [Errno 2] No such file or directory: 'autism_screening.csv'

In [2]:
pwd

'C:\\Users\\USER\\OneDrive - Al Akhawayn University in Ifrane\\Desktop\\ML'

In [ ]:
import tkinter as tk
from tkinter import messagebox
import pickle
import numpy as np

# Load the trained model
with open('autism_model.pkl', 'rb') as f:
    model = pickle.load(f)

def predict_asd():
    try:
        # Collect inputs from the UI
        inputs = []
        for i in range(10):
            inputs.append(int(score_vars[i].get()))
        
        inputs.append(float(age_entry.get()))
        inputs.append(1 if gender_var.get() == "Male" else 0)
        inputs.append(1 if jaundice_var.get() == "Yes" else 0)
        inputs.append(1 if family_var.get() == "Yes" else 0)

        # Prepare for prediction
        features = np.array([inputs])
        prediction = model.predict(features)

        # Show result
        result = "YES (Potential ASD Traits)" if prediction[0] == 1 else "NO (No ASD Traits detected)"
        messagebox.showinfo("Prediction Result", f"The screening result is: {result}")
    except Exception as e:
        messagebox.showerror("Error", f"Invalid input: {e}")

# Setup Tkinter Window
root = tk.Tk()
root.title("Autism Screening Tool")
root.geometry("400x600")

# --- UI Elements ---
tk.Label(root, text="Autism Screening (Adults)", font=("Arial", 14, "bold")).pack(pady=10)

# A1-A10 Score Checkboxes
tk.Label(root, text="Answer '1' if the trait is present, '0' if not:").pack()
score_vars = []
score_frame = tk.Frame(root)
score_frame.pack()
for i in range(10):
    var = tk.StringVar(value="0")
    score_vars.append(var)
    tk.Label(score_frame, text=f"A{i+1}:").grid(row=i//2, column=(i%2)*2)
    tk.Entry(score_frame, textvariable=var, width=5).grid(row=i//2, column=(i%2)*2+1, padx=10)

# Age
tk.Label(root, text="Age:").pack(pady=5)
age_entry = tk.Entry(root)
age_entry.insert(0, "25")
age_entry.pack()

# Gender
tk.Label(root, text="Gender:").pack(pady=5)
gender_var = tk.StringVar(value="Male")
tk.OptionMenu(root, gender_var, "Male", "Female").pack()

# Jaundice
tk.Label(root, text="Born with Jaundice?").pack(pady=5)
jaundice_var = tk.StringVar(value="No")
tk.OptionMenu(root, jaundice_var, "Yes", "No").pack()

# Family History
tk.Label(root, text="Family History of PDD/Autism?").pack(pady=5)
family_var = tk.StringVar(value="No")
tk.OptionMenu(root, family_var, "Yes", "No").pack()

# Predict Button
tk.Button(root, text="Predict Result", command=predict_asd, bg="blue", fg="white", font=("Arial", 10, "bold")).pack(pady=20)

root.mainloop()